# Run Experiments

Batch-runs the config overrides listed in `EXPERIMENTS` below (one file per experiment under `config/experiment/`) against a shared train/val/test split, each into its own timestamped `experiments/` run folder.

In [ ]:
import sys
import gc
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import torch

from omegaconf import OmegaConf, open_dict
from hydra import initialize, compose

current_dir = Path.cwd()
if 'notebooks' in current_dir.parts:
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.training.trainer import DGDTrainer
from src.data import create_dataloaders, get_sample_batches, save_sample_batches, collect_all_labels
from src.visualization.report import generate_training_figures
from src.utils import setup_device, set_random_seed, setup_cuml_acceleration

device = setup_device(verbose=True)
setup_cuml_acceleration(verbose=True)


In [ ]:
# Which experiments to run, in order -- matches filenames under
# config/experiment/ (without the .yaml extension). Comment out or remove
# entries to run a subset.
EXPERIMENTS = [
    "lower_noise_start",
]

# Composed once -- hydra's initialize() raises GlobalHydraException if
# entered a second time in the same kernel, so every experiment below reuses
# this same base config via OmegaConf.merge rather than a fresh compose().
with initialize(version_base=None, config_path="../config"):
    base_config = compose(config_name="config")

base_config.data.root_dir = str(project_root / "data")
base_config.paths.experiments_dir = str(project_root / "experiments")

print(f"Base config composed. Will run {len(EXPERIMENTS)} experiment(s): {EXPERIMENTS}")


In [ ]:
# Dataloaders and the visualization sample batch are shared across every
# experiment below: none of the experiment overrides touch data.* or
# random_seed, so the train/val/test split is identical for all of them --
# building it once here (instead of per experiment) is both faster and a
# stronger guarantee that every experiment is compared on the same data.
data_dir = project_root / "data"
data_dir.mkdir(exist_ok=True)

set_random_seed(seed=base_config.random_seed, device=device)
train_loader, val_loader, test_loader, class_names = create_dataloaders(base_config)

print(f"- Train loader: {len(train_loader)} batches")
print(f"- Val loader: {len(val_loader)} batches")
print(f"- Test loader (held out, unused here): {len(test_loader)} batches")
print(f"- Classes: {class_names}")

sample_data = get_sample_batches(train_loader, val_loader, device=device, n_per_class=5, n_classes=len(class_names))

samples_dir = project_root / "data" / "samples"
samples_dir.mkdir(exist_ok=True)
save_sample_batches(sample_data, str(samples_dir / "visualization_samples.pt"))

train_labels = collect_all_labels(train_loader)
val_labels = collect_all_labels(val_loader)


In [ ]:
results_summary = {}

for exp_name in EXPERIMENTS:
    print(f"\n{'='*70}")
    print(f"Experiment: {exp_name}")
    print(f"{'='*70}")

    trainer = None
    train_results = None

    try:
        override_path = project_root / "config" / "experiment" / f"{exp_name}.yaml"
        override = OmegaConf.load(override_path)
        # hydra's compose() leaves base_config in struct mode, which raises
        # ConfigKeyError for any key an override introduces that isn't
        # already present in the base schema (e.g. dist_params.mean/cov for
        # an override that switches distribution away from the default
        # uniform_ball's dist_params: {radius: ...}). open_dict relaxes
        # struct mode for just this merge.
        with open_dict(base_config):
            config = OmegaConf.merge(base_config, override)

        # Fresh seed per experiment: decoder/representation/GMM
        # initialization and noise draws should all start from the same
        # seed for every experiment, not drift across the loop.
        set_random_seed(seed=config.random_seed, device=device)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = Path(config.paths.experiments_dir) / f"{timestamp}_{config.experiment_name}"
        run_dir.mkdir(parents=True, exist_ok=True)

        with open_dict(config):
            config.paths.models_dir = str(run_dir / "models")
            config.paths.figures_dir = str(run_dir / "figures")

        OmegaConf.save(config, str(run_dir / "config.yaml"))
        print(f"Run directory: {run_dir}")
        print(f"Description: {config.get('description', '(none)')}")

        trainer = DGDTrainer(config=config, device=device, verbose=True)
        train_results = trainer.train(
            train_loader=train_loader,
            val_loader=val_loader,
            sample_data=sample_data,
            class_names=class_names,
        )

        generate_training_figures(
            experiment_dir=Path(config.paths.models_dir),
            figures_dir=Path(config.paths.figures_dir),
            class_names=class_names,
            train_labels=train_labels,
            val_labels=val_labels,
            sample_data=sample_data,
            device=device,
        )

        results_summary[exp_name] = {
            'status': 'completed',
            'run_dir': str(run_dir),
            'final_train_loss': train_results['final_train_loss'],
            'final_val_loss': train_results['final_val_loss'],
            'best_train_loss': train_results['best_train_loss'],
            'best_epoch': train_results['best_epoch'],
        }
        print(f"Completed: final_val_loss={train_results['final_val_loss']:.4f}, "
              f"best_epoch={train_results['best_epoch']}/{config.training.epochs}")

    except Exception as e:
        print(f"FAILED: {exp_name}: {type(e).__name__}: {e}")
        results_summary[exp_name] = {'status': 'failed', 'error': f"{type(e).__name__}: {e}"}

    finally:
        # Free GPU memory before the next experiment -- each run builds a
        # fresh decoder + train/val representation layers + GMM, and five
        # sequential 200-epoch runs will otherwise stack memory across the
        # whole loop.
        if train_results is not None:
            del train_results
        if trainer is not None:
            del trainer
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n{'='*70}")
print("Batch complete.")


## Summary

Quick side-by-side comparison of the batch. For a deeper look at any one experiment (latent space, reconstructions, noise diagnostics, GMM cluster breakdown), open its `figures/` folder under the run directory printed above, or point `dgd_test_inference.ipynb` / `noise_injection_explained.ipynb` at it (both auto-discover the newest completed run, so run the one you want to inspect last, or edit their run-selection cell to pick a specific `run_dir`).


In [ ]:
print(f"{'Experiment':<22} {'Status':<10} {'Final Val Loss':<16} {'Best Train Loss':<17} {'Best Epoch'}")
print("-" * 80)
for exp_name, r in results_summary.items():
    if r['status'] == 'completed':
        print(f"{exp_name:<22} {r['status']:<10} {r['final_val_loss']:<16.4f} "
              f"{r['best_train_loss']:<17.4f} {r['best_epoch']}")
    else:
        print(f"{exp_name:<22} {r['status']:<10} {r.get('error', '')}")
